Run the next cell first. In Colab, add the course key under the key
icon in the left sidebar as `COURSE_API_KEY`, with *Notebook access* on.

In [ ]:
# Course setup. In Colab: add the course key under the key icon (left sidebar)
# as COURSE_API_KEY. On your own machine it uses Ollama instead.
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Pinned, including the transitive ones. Unpinned, pip takes whatever
    # shipped this morning : a newer core moves ModelError, and a newer openai
    # rejects this endpoint's usage payload. Keep in step with requirements.txt.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "langchain==1.3.9", "langchain-core==1.4.7",
                    "langchain-openai==1.3.2", "langchain-classic==1.0.8",
                    "langgraph==1.2.5", "openai==2.41.1", "python-dotenv"],
                   check=True)
    MODEL = "qwen3.8-flash"          # try qwen3.8-max too
    BASE = "https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1"
    THINKING = {"enable_thinking": False}
    KEY = os.getenv("COURSE_API_KEY")
    if not KEY:
        try:
            from google.colab import userdata
            KEY = userdata.get("COURSE_API_KEY")   # raises if unset or not shared
        except Exception:
            import getpass
            KEY = getpass.getpass("Course key (paste the one from the trainer): ")
else:
    from dotenv import load_dotenv
    load_dotenv()
    MODEL = "qwen3.5:2b"             # try qwen3.5:4b too
    BASE = "http://localhost:11434/v1"
    KEY = "ollama"
    THINKING = {"reasoning_effort": "none"}

from langchain_openai import ChatOpenAI


def make_llm(**kw):
    kw.setdefault("temperature", 0)
    kw.setdefault("model", MODEL)
    kw.setdefault("extra_body", THINKING)
    return ChatOpenAI(base_url=BASE, api_key=KEY, **kw)


def make_embeddings(**kw):
    # The course endpoint has no embeddings, so they run here instead. 90 MB,
    # installed only by the notebooks that actually ask for them.
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "langchain-huggingface==0.3.1"], check=True)
        from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


DATA_URL = ("https://raw.githubusercontent.com/FeikoWielsma/"
            "Building-AI-Agents/main/data/")


def data(name):
    """Path to a course data file. Downloads it in Colab, local copy otherwise."""
    if os.path.exists(f"../data/{name}"):
        return f"../data/{name}"
    if not os.path.exists(name):
        import urllib.request
        urllib.request.urlretrieve(DATA_URL + name, name)
    return name


llm = make_llm()
print(f"model={MODEL}")

## Vector databases

- An embedding turns text into a list of numbers standing for its meaning
- Texts that mean similar things get similar numbers
- So "find the closest numbers" becomes "find the most similar text"

Embeds a handful of short texts, stores them, and queries for the nearest.

### Exercise 
A simple vector store and query. Modify the code below in the indicated places.
Replace the list of texts with our own - these could be, for example, article content copied from any source on topics that interest us.
Modify the search texts so that one matches the topic of one of the added texts and the other matches a different one. Use different words in the search texts than those contained in the article fragments.

In [ ]:
%pip install -q langchain-ollama faiss-cpu numpy

#### Optional : write it first

The next cell is the finished version ; nothing below depends on doing this first.

```python
from langchain_openai import OpenAIEmbeddings
import faiss
import numpy as np

# Ollama embedding model (replaces SentenceTransformer)
embedder = make_embeddings()

sentences = [  # modify the list of texts for semantic matching
    "dinosaurs live in africa but in different time dimension",
    "this is sentence about little cat that liked to eat fast food",
    "this is the another sample sentence which is here just to not be matched while other one is"
]

embeddings = np.array(embedder.embed_documents(sentences), dtype="float32")
d = embeddings.shape[1]        # dimensions (nomic-embed-text = 768)

# Build index
index = faiss.IndexFlatL2(d)
index.add(embeddings)

queryText = "____"  # <- modify the search text (match one sentence, different words)
q = np.array(embedder.embed_query(queryText), dtype="float32").reshape(1, -1)
_, idx = index.search(q, 1)
print(queryText + " matches:\n" + sentences[idx[0][0]])

queryText = "____"  # <- modify the search text (match a different sentence)
q = np.array(embedder.embed_query(queryText), dtype="float32").reshape(1, -1)
_, idx = index.search(q, 1)
print(queryText + " matches:\n" + sentences[idx[0][0]])
```

In [ ]:
from langchain_openai import OpenAIEmbeddings
import faiss
import numpy as np

embedder = make_embeddings()

# Own texts : two clearly different topics (space, cooking) plus a filler
sentences = [
    "Astronomers discovered a new exoplanet orbiting a distant star in the Milky Way.",
    "The chef slowly simmered the tomato sauce for hours to deepen its flavour.",
    "This is an unrelated filler sentence that should not match either query."
]

embeddings = np.array(embedder.embed_documents(sentences), dtype="float32")
d = embeddings.shape[1]

index = faiss.IndexFlatL2(d)
index.add(embeddings)

# Query 1: about space : uses NONE of the words in the space sentence
queryText = "planets and galaxies in outer space"
q = np.array(embedder.embed_query(queryText), dtype="float32").reshape(1, -1)
_, idx = index.search(q, 1)
print(queryText + " matches:\n" + sentences[idx[0][0]] + "\n")

# Query 2: about cooking : uses NONE of the words in the cooking sentence
queryText = "preparing a delicious italian meal in the kitchen"
q = np.array(embedder.embed_query(queryText), dtype="float32").reshape(1, -1)
_, idx = index.search(q, 1)
print(queryText + " matches:\n" + sentences[idx[0][0]])

### Try a query with no words in common

- Ask for something using entirely different wording from the stored text
- A keyword search would find nothing. This finds it, and that difference is
  the whole reason vector stores exist